# ការទទួលស្គាល់អង្គភាពឈ្មោះ (NER)

កំណត់ត្រានេះមកពី [កម្មវិធីសិក្សា AI សម្រាប់អ្នកចាប់ផ្តើម](http://aka.ms/ai-beginners)។

នៅក្នុងឧទាហរណ៍នេះ យើងនឹងរៀនពីរបៀបបណ្តុះបណ្តាលម៉ូដែល NER លើ [Annotated Corpus for Named Entity Recognition](https://www.kaggle.com/datasets/abhinavwalia95/entity-annotated-corpus) Dataset ពី Kaggle។ មុនពេលបន្ត សូមទាញយកឯកសារ [ner_dataset.csv](https://www.kaggle.com/datasets/abhinavwalia95/entity-annotated-corpus?resource=download&select=ner_dataset.csv) ទៅក្នុងថតបច្ចុប្បន្ន។


In [62]:
import pandas as pd
from tensorflow import keras
import numpy as np

## ការរៀបចំឯកទិន្នន័យ

យើងនឹងចាប់ផ្តើមដោយការអានឯកទិន្នន័យទៅក្នុង dataframe។ ប្រសិនបើអ្នកចង់រៀនបន្ថែមអំពីការប្រើប្រាស់ Pandas សូមចូលទៅកាន់ [មេរៀនអំពីការដំណើរការទិន្នន័យ](https://github.com/microsoft/Data-Science-For-Beginners/tree/main/2-Working-With-Data/07-python) នៅក្នុង [វិទ្យាសាស្ត្រ​ទិន្នន័យ​សម្រាប់​អ្នក​ចាប់ផ្តើម](http://aka.ms/datascience-beginners) របស់យើង។


In [3]:
df = pd.read_csv('ner_dataset.csv',encoding='unicode-escape')
df.head()

,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,NaN,of,IN,O
2,NaN,demonstrators,NNS,O
3,NaN,have,VBP,O
4,NaN,marched,VBN,O


មកយកស្លាកផ្ទាល់ខ្លួន ហើយបង្កើតវចនានុក្រមស្វែងរកដែលយើងអាចប្រើដើម្បីបម្លែងស្លាកទៅជាលេខថ្នាក់:


In [4]:
tags = df.Tag.unique()
tags

array(['O', 'B-geo', 'B-gpe', 'B-per', 'I-geo', 'B-org', 'I-org', 'B-tim',
       'B-art', 'I-art', 'I-per', 'I-gpe', 'I-tim', 'B-nat', 'B-eve',
       'I-eve', 'I-nat'], dtype=object)

In [8]:
id2tag = dict(enumerate(tags))
tag2id = { v : k for k,v in id2tag.items() }

id2tag[0]

'O'

ឥឡូវនេះយើងត្រូវធ្វើដូចគ្នានឹងវ៉ុកាបសារ។ សម្រាប់ភាពសាមញ្ញ យើងនឹងបង្កើតវ៉ុកាបសារដោយមិនគិតគូសអាករណ៍នៃពាក្យឡើយ; ក្នុងជីវិតពិត អ្នកអាចចង់ប្រើកម្មវិធីផ្សារជា vectorizer របស់ Keras ហើយកំណត់ចំនួនពាក្យ។


In [14]:
vocab = set(df['Word'].apply(lambda x: x.lower()))
id2word = { i+1 : v for i,v in enumerate(vocab) }
id2word[0] = '<UNK>'
vocab.add('<UNK>')
word2id = { v : k for k,v in id2word.items() }

យើងត្រូវការបង្កើតឌាតាសែតនៃប្រយោគសម្រាប់ការបណ្តុះបណ្តាល។ ម្តងនេះយើងចូលរួមតាមរយៈឌាតាសែតដើមហើយបំបែកប្រយោគមួយៗចេញទៅជា `X` (បញ្ជីពាក្យ) និង `Y` (បញ្ជីតុកខេន):


In [41]:
X,Y = [],[]
s,t = [],[]
for i,row in df[['Sentence #','Word','Tag']].iterrows():
    if pd.isna(row['Sentence #']):
        s.append(row['Word'])
        t.append(row['Tag'])
    else:
        if len(s)>0:
            X.append(s)
            Y.append(t)
        s,t = [row['Word']],[row['Tag']]
X.append(s)
Y.append(t)


ឥឡូវនេះយើងនឹងបំលែងពាក្យ និងតូចតាមទាំងអស់ទៅជាវ៉ិចទ័រ៖


In [93]:
def vectorize(seq):
    return [word2id[x.lower()] for x in seq]

def tagify(seq):
    return [tag2id[x] for x in seq]

Xv = list(map(vectorize,X))
Yv = list(map(tagify,Y))

Xv[0], Yv[0]

([10386,
  23515,
  4134,
  29620,
  7954,
  13583,
  21193,
  12222,
  27322,
  18258,
  5815,
  15880,
  5355,
  25242,
  31327,
  18258,
  27067,
  23515,
  26444,
  14412,
  358,
  26551,
  5011,
  30558],
 [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0])

សម្រាប់ភាពងាយស្រួល យើងនឹងបញ្ចូលសញ្ញាសូន្យ 0 ទៅក្នុងប្រយោគទាំងអស់ដើម្បីឲ្យបានប្រវែងអតិបរមា។ ក្នុងជីវិតពិតប្រាកដ យើងអាចចង់ប្រើយុទ្ធសាស្រ្តឆ្លាតវៃជាងនេះ ហើយបញ្ចូលសញ្ញា 0 ក្នុងលំដាប់តែមួយនៅក្នុងមីនីបាច់ប៉ុណ្ណោះ។


In [51]:
X_data = keras.preprocessing.sequence.pad_sequences(Xv,padding='post')
Y_data = keras.preprocessing.sequence.pad_sequences(Yv,padding='post')

## ការបញ្ជាក់បណ្តាញចាត់ថ្នាក់ Token

យើងនឹងប្រើបណ្ដាញ LSTM ជាន់ពីរដែរដោយទិសពីរដើម្បីចាត់ថ្នាក់ token។ ដើម្បីអាចអនុវត្តកម្មវិធីចាត់ថ្នាក់ដិតសម្រាប់លទ្ធផលនៃជាន់ចុងក្រោយរបស់ LSTM នីមួយៗ យើងនឹងប្រើ `TimeDistributed` ដែលធ្វើការចម្លងល័កស្ទរដិតដូចគ្នាទៅលើលទ្ធផលនៃ LSTM ក្នុងជំហាននីមួយៗ៖ 


In [94]:
maxlen = X_data.shape[1]
vocab_size = len(vocab)
num_tags = len(tags)
model = keras.models.Sequential([
    keras.layers.Embedding(vocab_size, 300, input_length=maxlen),
    keras.layers.Bidirectional(keras.layers.LSTM(units=100, activation='tanh', return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(units=100, activation='tanh', return_sequences=True)),
    keras.layers.TimeDistributed(keras.layers.Dense(num_tags, activation='softmax'))
])
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['acc'])
model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_4 (Embedding)     (None, 104, 300)          9545400   
                                                                 
 bidirectional_6 (Bidirectio  (None, 104, 200)         320800    
 nal)                                                            
                                                                 
 bidirectional_7 (Bidirectio  (None, 104, 200)         240800    
 nal)                                                            
                                                                 
 time_distributed_3 (TimeDis  (None, 104, 17)          3417      
 tributed)                                                       
                                                                 
Total params: 10,110,417
Trainable params: 10,110,417
Non-trainable params: 0
__________________________________________

ចំណាំនៅទីនេះថា យើងកំពុងបញ្ជាក់យ៉ាងច្បាស់ `maxlen` សម្រាប់ឃ្លាំងទិន្នន័យរបស់យើង - បើសិនជាយើងចង់ឲ្យបណ្តាញអាចដោះស្រាយអំពីលំដាប់ប្រវែងខុសៗគ្នា ត្រូវការ​ឲ្យយើងមានប្រាជ្ញា​បន្តិចនៅពេលកំណត់បណ្តាញ។

ឥឡូវនេះ យើងចាប់ផ្តើមបណ្តុះម៉ូដែល។ សម្រាប់ល្បឿន យើងនឹងបណ្តុះតែមួយវគ្គ តែក៏អ្នកអាចព្យាយាមបណ្តុះរយៈពេលយូរជាងនេះបាន។ លើសពីនេះ អ្នកអាចចែកជាមួយផ្នែកមួយនៃឃ្លាំងទិន្នន័យជាឃ្លាំងបណ្តុះ ដើម្បីមើលភាពត្រឹមត្រូវក្នុងការត្រួតពិនិត្យបានផង។


In [57]:
model.fit(X_data,Y_data)

1499/1499 [==============================] - 740s 488ms/step - loss: 0.0667 - acc: 0.9841


## ការធ្វើតេស្តលទ្ធផល

ឥឡូវនេះអោយយើងមើលថាតើម៉ូដែលការទទួលស្គាល់អង្គភាពរបស់យើងដំណើរការយ៉ាងដូចម្តេចលើប្រយោគគំរូមួយ:


In [91]:
sent = 'John Smith went to Paris to attend a conference in cancer development institute'
words = sent.lower().split()
v = keras.preprocessing.sequence.pad_sequences([[word2id[x] for x in words]],padding='post',maxlen=maxlen)
res = model(v)[0]

In [92]:
r = np.argmax(res.numpy(),axis=1)
for i,w in zip(r,words):
    print(f"{w} -> {id2tag[i]}")

john -> B-per
smith -> I-per
went -> O
to -> O
paris -> B-geo
to -> O
attend -> O
a -> O
conference -> O
in -> O
cancer -> B-org
development -> I-org
institute -> I-org


## Takeaway

ម៉ូដែល LSTM សាមញ្ញមួយ បង្ហាញលទ្ធផលសមរម្យនៅក្នុង NER។ ទោះជាយ៉ាងណា ដើម្បីទទួលបានលទ្ធផលល្អជាងនេះ អ្នកប្រហែលចង់ប្រើម៉ូដែលភាសាធំដែលបានបណ្តុះបណ្តាលរួចដូចជាវិធី BERT។ ការបណ្តុះបណ្តាល BERT សម្រាប់ NER ដោយប្រើបណ្ណាល័យ Huggingface Transformers ត្រូវបានពិពណ៌នាអាស្រ័យ [នៅទីនេះ](https://huggingface.co/course/chapter7/2?fw=pt)।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**៖  
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ខណៈពេលដែលយើងខិតខំរកសុពលភាព សូមយល់ថាការបកប្រែដោយស្វ័យប្រវត្តិបំពាក់ដោយកំហុសឬការខ្វះត្រឹមត្រូវខ្លះៗ។ ឯកសារដើមជាភាសាបុរាណគួរត្រូវបានគេយកទៅជាដើមហើយមានអំណាច។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមផ្តួចផ្តើមការបកប្រែដោយអ្នកជំនាញមនុស្ស។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំឬការបកស្រាយខុសពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
